<a href="https://colab.research.google.com/github/Toan-ten/Arxiv-Project-Group10/blob/integration%2Fm2-safe-plus-fixed/notebooks/m2/m2_team_colab_export.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# Group 10 — M2 Team Dataset Export (CPU-only)
Một notebook cho cả 4 thành viên. Chỉ đọc M1 và xuất dataset M2; không crawl,
không gọi S2, không upload tự động, không phải final benchmark.
Parser cố định `072ee384efa2e22f743ada7786aa0f67113a1490`.
Helper được tải từ release SHA riêng trên GitHub, không cần file trên máy Windows.

| STUDENT_ID | Cohorts được giao |
|---|---|
| 24120010 | 0001–0050 |
| 24120035 | 0051–0100 |
| 24120232 | 0101–0150 |
| 24120260 | 0151–0200 |

Chọn Runtime → Change runtime type → **CPU**. Mỗi thành viên chạy runtime độc lập;
chỉ **một writer** cho mỗi campaign. Không dùng Run all trước khi kiểm tra smoke.


In [ ]:
# 1. Cấu hình duy nhất cần chỉnh. Giữ MSSV dạng string; cohort dạng số nguyên.
from pathlib import Path
STUDENT_ID = "24120232"
START_COHORT = 101
END_COHORT = 150
BATCH_SIZE = 5
INPUT_DRIVE_DIR = f"/content/drive/MyDrive/M1-pipeline-optimization-full/{STUDENT_ID}/archives"
OUTPUT_DRIVE_DIR = f"/content/drive/MyDrive/ArxivGroup10/{STUDENT_ID}/m2-exports"
HF_REPO_ID = "toanten/arxiv-project-group10"


## 2. Setup — chạy lại sau mỗi runtime restart
INPUT_DRIVE_DIR mặc định là `MyDrive/M1-pipeline-optimization-full/<STUDENT_ID>/archives`.
Thư mục phải tồn tại; notebook không tự tạo input. Cả 4 thành viên dùng cùng cấu trúc này.
Trước khi chạy, cần **ba file cho mỗi cohort** trong INPUT_DRIVE_DIR:
`cohort_NNNN_corpus.tar.gz`, `cohort_NNNN_corpus.manifest.json`,
`cohort_NNNN_corpus.tar.gz.sha256`. Có thể chép trước cohort đầu để smoke;
chỉ batch đang chạy cần đủ input. Không giả định đủ dữ liệu cho cả dải khi chỉ có một gói.

Setup mount Drive, tải helper và parser ở **hai SHA cố định**, cài dependency M2,
kiểm tra MSSV/dải cohort và cho biết campaign path. Không cần API key hoặc HF token.
Không đổi các SHA để ép dùng checkpoint cũ. Nếu campaign tồn tại, cài lại đúng Arrow version đã ghi.

Trước Smoke/Run batch/Audit, preflight kiểm tra đủ ba file cho phạm vi đang chạy;
thiếu file sẽ liệt kê đường dẫn đầy đủ và dừng trước khi gọi M2.


In [ ]:
from google.colab import drive
import json, os, shutil, subprocess, sys
if sys.version_info < (3, 11):
    raise RuntimeError("C?n Python 3.11 tr? l?n")
PARSER_COMMIT = "072ee384efa2e22f743ada7786aa0f67113a1490"
HELPER_COMMIT = "13fb8c693382694d7eaef976002c1bf2aa1e88de"
REPO_URL = "https://github.com/Toan-ten/Arxiv-Project-Group10.git"
CODE_ROOT = Path("/content/m2-parser-" + PARSER_COMMIT[:12])
HELPER_ROOT = Path("/content/m2-export-helper-" + HELPER_COMMIT[:12])
drive.mount("/content/drive")
if not Path(INPUT_DRIVE_DIR).is_dir():
    first = f"cohort_{START_COHORT:04d}_corpus"
    required = [str(Path(INPUT_DRIVE_DIR) / (first + suffix))
                for suffix in (".tar.gz", ".manifest.json", ".tar.gz.sha256")]
    raise FileNotFoundError(f"Input directory does not exist: {INPUT_DRIVE_DIR}; required files: " + ", ".join(required))
gpu = False
if shutil.which("nvidia-smi"):
    probe = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
    gpu = probe.returncode == 0 and bool(probe.stdout.strip())
if os.environ.get("COLAB_TPU_ADDR") or os.environ.get("COLAB_GPU") == "1" or gpu:
    raise RuntimeError("Hãy đổi runtime sang CPU rồi chạy lại Setup")

def checkout_release(target, commit):
    if not target.exists():
        target.mkdir(parents=True)
        subprocess.run(["git", "init", str(target)], check=True, stdout=subprocess.DEVNULL)
        subprocess.run(["git", "-C", str(target), "remote", "add", "origin", REPO_URL], check=True)
        subprocess.run(["git", "-C", str(target), "fetch", "--depth", "1", "origin", commit], check=True)
        subprocess.run(["git", "-C", str(target), "checkout", "--detach", commit], check=True)
    actual = subprocess.check_output(["git", "-C", str(target), "rev-parse", "HEAD"], text=True).strip()
    if actual != commit:
        raise RuntimeError("Checkout khác SHA yêu cầu; không tự reset dữ liệu hiện có")
    subprocess.run(["git", "-C", str(target), "diff", "--exit-code", commit, "--"], check=True)

checkout_release(CODE_ROOT, PARSER_COMMIT)
checkout_release(HELPER_ROOT, HELPER_COMMIT)
# Chỉ import helper (stdlib) lúc này; chưa import parser/pyarrow.
import importlib.util
spec = importlib.util.spec_from_file_location("m2_export_tools", HELPER_ROOT / "scripts/m2_colab_export.py")
export_tools = importlib.util.module_from_spec(spec)
spec.loader.exec_module(export_tools)
config = dict(student_id=STUDENT_ID, start_cohort=START_COHORT, end_cohort=END_COHORT,
              batch_size=BATCH_SIZE, input_drive_dir=str(INPUT_DRIVE_DIR),
              output_drive_dir=str(OUTPUT_DRIVE_DIR), hf_repo_id=HF_REPO_ID,
              repo=str(CODE_ROOT), helper_commit=HELPER_COMMIT,
              scratch="/content/m2-input")
scope = export_tools.scope_for(config)
CAMPAIGN = export_tools.campaign_path(config, scope)
identity_path = CAMPAIGN / "campaign.json"
if identity_path.exists():
    identity = export_tools.read(identity_path)
    if (identity["parser_commit"] != PARSER_COMMIT or identity["helper_commit"] != HELPER_COMMIT
            or identity["python_major_minor"] != list(sys.version_info[:2])):
        raise RuntimeError("Campaign dùng parser/helper/Python khác; dừng để review")
    dependency = ["pyarrow==" + identity["pyarrow"]]
else:
    dependency = ["-r", str(CODE_ROOT / "requirements-m2.txt")]
subprocess.run([sys.executable, "-m", "pip", "install", *dependency], check=True)
CONFIG_PATH = Path("/content") / ("m2-export-config-" + STUDENT_ID + ".json")
CONFIG_PATH.write_text(json.dumps(config, indent=2), encoding="utf8")
available, missing = [], []
for name in scope["cohorts"]:
    present = all((Path(INPUT_DRIVE_DIR) / (name + suffix)).is_file()
                  for suffix in (".tar.gz", ".manifest.json", ".tar.gz.sha256"))
    (available if present else missing).append(name)
print(dict(parser_commit=PARSER_COMMIT, helper_commit=HELPER_COMMIT, student_id=STUDENT_ID,
           cohorts_requested=len(scope["cohorts"]), input_triplets_present=len(available),
           missing_inputs=missing, campaign=str(CAMPAIGN), hf_repo_id=HF_REPO_ID,
           purpose="data_export_not_final_benchmark", scratch_free_GiB=round(shutil.disk_usage("/content").free/1024**3, 2)))

def require_input_files(cohorts):
    root = Path(INPUT_DRIVE_DIR)
    if not root.is_dir():
        required = [str(root / (name + suffix)) for name in cohorts
                    for suffix in (".tar.gz", ".manifest.json", ".tar.gz.sha256")]
        raise FileNotFoundError(f"Input directory does not exist: {root}; required files: " + ", ".join(required))
    missing = [root / (name + suffix) for name in cohorts
               for suffix in (".tar.gz", ".manifest.json", ".tar.gz.sha256")
               if not (root / (name + suffix)).is_file() or (root / (name + suffix)).is_symlink()]
    if missing:
        raise FileNotFoundError("Missing or unsafe corpus input files:\n" + "\n".join(str(p) for p in missing))


def active_input_cohorts(action):
    names = scope["cohorts"]
    batches = [names[i:i + BATCH_SIZE] for i in range(0, len(names), BATCH_SIZE)]
    def complete(batch):
        label = batch[0][7:11] + "-" + batch[-1][7:11]
        return (CAMPAIGN / "batches" / (label + ".json")).is_file()
    if action == "smoke":
        return names[:1]
    if action == "next":
        return next((batch for batch in batches if not complete(batch)), [])
    if action == "audit":
        return [name for batch in batches if complete(batch) for name in batch]
    raise ValueError(f"Unknown export action: {action}")


def run_export(action):
    require_input_files(active_input_cohorts(action))
    import signal
    command = [sys.executable, "-X", "utf8", "-B", str(HELPER_ROOT / "scripts/m2_colab_export.py"),
               "--config", str(CONFIG_PATH), "--action", action]
    env = dict(os.environ, PYTHONDONTWRITEBYTECODE="1", CUDA_VISIBLE_DEVICES="")
    process = subprocess.Popen(command, env=env, start_new_session=True, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        result = process.wait()
    except KeyboardInterrupt:
        try:
            os.killpg(process.pid, signal.SIGINT)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
        print("Đã dừng invocation; giữ checkpoint. Kiểm tra writer lock trước khi Resume.")
        raise
    finally:
        process.stdout.close()
    if result:
        raise RuntimeError("Export dừng an toàn; đọc lỗi phía trên. Không tự retry vô hạn.")


## 3. Smoke — toàn bộ cohort đầu trong cấu hình
Mọi version của một publication luôn được xử lý cùng nhau. Xem statuses và row counts;
`partial_coverage=true` không phải hoàn tất dữ liệu. Chạy lại smoke trên cohort verified chỉ audit.


In [ ]:
run_export("smoke")


## 4. Run batch / Resume
Mỗi lần Run xử lý batch chưa hoàn tất đầu tiên, tối đa BATCH_SIZE cohorts; batch cuối có thể ít hơn.
Cohort đã smoke được dùng lại. Sau restart, chạy cấu hình → Setup → Run batch với cùng cấu hình.
Stable parse errors được giữ trong coverage; timeout còn retryable không có completion marker.
Không tự phá lock còn lại sau hard kill. Không chạy hai runtime trên cùng campaign.


In [ ]:
run_export("next")


## 5. Audit lại batches hoàn tất
Kiểm tra lại checksum input, checkpoint, schema/rows Parquet, hierarchy đúng version,
citation và bibliography evidence cùng version. Không parse lại cohort đã verified.
Nếu phiên bản thiếu entry/file hoặc semantics chưa hỗ trợ, giữ lỗi rõ ràng, không tạo records giả.


In [ ]:
run_export("audit")


## 6. Publish thủ công các batch đã verified
Chỉ chọn gói có READY.json sau Audit thành công, dưới `CAMPAIGN/upload/`.
Mỗi gói có ba thư mục Parquet, coverage.json, checksums.json và BATCH_CARD.md.
Có thể upload một batch trước khi đủ dải, nhưng phải ghi **partial coverage**.

Mở `https://huggingface.co/datasets/<HF_REPO_ID>` → Files and versions → Add file → Upload files.
Dùng prefix riêng `m2/<STUDENT_ID>/072ee384efa2/<batch>/`, không upload trùng batch hoặc overwrite dữ liệu người khác.
Giữ `data/elements/`, `data/hierarchy/`, `data/bibtex/` riêng và cấu hình ba dataset configs khác nhau.
Không thay README chung bằng BATCH_CARD; không upload raw M1, checkpoint làm việc hoặc secrets.
Notebook không dùng HF token và không upload tự động.

Xem `docs/m2_colab_export.md` trong cùng repository để biết schema, writer locks,
source uppercase, thiếu BBL artifacts và hướng dẫn upload. Lượt export không thay final benchmark.
